In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("Groq Key Loaded:", GROQ_API_KEY is not None)


Groq Key Loaded: True


In [2]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)


c:\Users\mihir\Documents\Python Scripts\MajorProj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from llama_index.llms.groq import Groq

Settings.llm = Groq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.0
)


In [4]:
import chromadb
persist_dir = "./chroma_db"

chroma_client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=chromadb.Settings(anonymized_telemetry=False)
)

try:
    #wipe old rag_collection data
    chroma_client.delete_collection("rag_collection")
except:
    # If the collection doesn't exist yet, just move on
    pass

collection = chroma_client.get_or_create_collection(name="rag_collection")



In [5]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [6]:
import fitz
from llama_index.core import Document
from llama_index.core.schema import TextNode
from llama_index.core.node_parser import SentenceSplitter

def translate_full_document(text_en):
    # Translate the entire document to Chinese using Groq.
    prompt = f"Translate the following technical document to Chinese, preserving all terminology:\n\n{text_en}"
    response = Settings.llm.complete(prompt)
    return str(response)

def process_aligned_ingestion(file_path):
    #Extraction
    doc = fitz.open(file_path)
    full_en_text = "\n".join([page.get_text() for page in doc])

    #Document-wide translation
    full_zh_text = translate_full_document(full_en_text)

    #Aligned Chunking
    en_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
    en_chunks = en_splitter.split_text(full_en_text)
    
    #Calculate ratio to map chinese text to English chunks
    nodes = []
    avg_zh_len = len(full_zh_text) // len(en_chunks)
    
    for i, en_chunk in enumerate(en_chunks):
        # Map corresponding chinese segment
        zh_segment = full_zh_text[i * avg_zh_len : (i + 1) * avg_zh_len]

        # Storage with chinese as auxiliary payload
        node = TextNode(
            text=en_chunk, # This gets embedded
            metadata={
                "zh_context": zh_segment, # For LLM reasoning
                "file_name": file_path.split("/")[-1]
            }
        )
        nodes.append(node)
    
    return nodes


In [7]:
from llama_index.core import PromptTemplate, VectorStoreIndex

file_path = "C:\\Users\\mihir\\OneDrive\\Documents\\Taliban.pdf"
nodes = process_aligned_ingestion(file_path)

index = VectorStoreIndex(nodes, storage_context=storage_context)

# Configure RAG with Chinese Reasoning 
qa_prompt_str = (
    "Context information is below, including a Chinese translation (zh_context) in metadata.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Reason using the Chinese context for accuracy and token efficiency, but answer in English.\n"
    "Query: {query_str}\n"
    "Answer: "
)
qa_prompt = PromptTemplate(qa_prompt_str)



query_engine = index.as_query_engine()
query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt})

# Execute Query
response = query_engine.query("Is the stem soft?")
print(response)

No, the stem is not soft. According to the text, the stem is "rough-hairy" and "erect rough-hairy", indicating that it has a rough texture.


In [29]:
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata

query_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="document_expert",
        description="Extracts info from the document using English/Chinese context."
    )
)
# ReAct Agent with system prompt
agent = ReActAgent(
    tools=[query_tool],
    llm=Settings.llm,
    verbose=True,
    # System prompt is passed directly to the constructor
    system_prompt=(
        "You are an expert analyst. Always prioritize the 'zh_context' "
        "in metadata for your internal reasoning, but answer in English."
    )
)

response =  await agent.run("Based on the document, is the stem soft? Verify using the Chinese context.")
print(response)

Yes, the stem of the common sunflower is soft in the sense that it is rough-hairy, but it is not a soft stem in the sense that it is not a flexible or pliable stem, rather it is a stiff stem that can reach heights of 3 meters.
